In [2]:
import pandas as pd
import numpy as np

### Ensure num records == num files

In [3]:
import os

reports = pd.read_csv('/Users/madhu/Desktop/WFP/wfp_reports/reports_metadata.csv')
counter = 0

def count_files_in_folders(base_path, counter):
    for root, dirs, files in os.walk(base_path):
        # Count only files (not subdirectories)
        num_files = len([f for f in files if os.path.isfile(os.path.join(root, f))])
        counter += num_files
    return counter
        # print(f"{root}: {num_files} file(s)")

# Example usage
base_directory = "/Users/madhu/Desktop/WFP/wfp_reports"
counter = count_files_in_folders(base_directory, counter)
reports.shape[0] == counter

False

In [4]:
reports.shape[0]

629

### Certain Reports seem to be repeated across multiple Categories

In [5]:
reports['Topic'] = reports['Topic'].str.replace(r'[^A-Za-z\s]', '', regex=True).str.strip()

In [6]:
repeated_reports = reports.groupby(['Report Name'])['Topic'].agg(unique_topics = lambda x : x.unique(),
                                                num_unique_topics = lambda x : x.nunique()).reset_index()

In [7]:
repeated_reports = repeated_reports[repeated_reports['num_unique_topics'] > 1].sort_values(['num_unique_topics'],
                                                                                           ascending = False)

In [14]:
for i in repeated_reports['unique_topics'][0:1]:
    print(i)

['Food security analysis VAM' 'Food Assistance' 'Inkind food distribution'
 'Nutrition' 'Food fortification' 'HIV and tuberculosis'
 'Specialized nutritious food' 'School meals'
 'Social Protection and Safety Nets' 'Food safety and quality'
 'Sustainable livelihoods and ecosystems']


In [15]:
repeated_reports = repeated_reports.assign(
    topics_exploded = repeated_reports["unique_topics"]
).explode("topics_exploded")


### Topic Pairs appearing together

In [22]:
from itertools import combinations
import pandas as pd

# Step 2: get all topic pairs per report
pairs = []
for topics in repeated_reports['unique_topics']:
    if len(topics) > 1:
        pairs.extend(list(combinations(sorted(topics), 2)))

# Step 3: count frequency of each topic pair
pair_counts = pd.Series(pairs).value_counts().reset_index()
pair_counts.columns = ['Topic_Pair', 'Count']

pair_counts.head(10)


,Topic_Pair,Count
0,"(Funding and donors, Monitoring evaluation and...",18
1,"(Anticipatory Action, Climate services)",7
2,"(Climate action, Climate services)",5
3,"(Smallholder agricultural market support, Soci...",4
4,"(Food Assistance for Assets, Sustainable livel...",4
5,"(Climate action, Climate change adaptation)",4
6,"(Anticipatory Action, Climate action)",4
7,"(Nutrition, Social Protection and Safety Nets)",4
8,"(Resilience programming, Sustainable livelihoo...",3
9,"(Food systems, Smallholder agricultural market...",3


In [24]:
reports['Topic'].nunique()

66

### PDF Data Extraction

In [28]:
import pymupdf  # PyMuPDF

def extract_pdf_structure(pdf_path):
    doc = pymupdf.open(pdf_path)
    pages_output = []

    for page in doc:
        blocks = page.get_text("dict")["blocks"]
        page_data = []

        for b in blocks:
            if "lines" not in b:
                continue
            text = ""
            for line in b["lines"]:
                for span in line["spans"]:
                    text += span["text"] + " "
                    font_size = span["size"]
            bbox = b["bbox"]

            page_data.append({
                "text": text.strip(),
                "bbox": bbox,
                "font_size": font_size,
                "page_num": page.number
            })

        pages_output.append(page_data)
    return pages_output

pdf_data = extract_pdf_structure("/Users/madhu/Desktop/WFP/wfp_reports/wfp_reports/Academia_and_think_tanks/2025_Anticipatory_Action_Learning_and_validation_R.pdf")
pdf_data[:2]  # inspect sample


[[{'text': 'Anticipatory Action  Learning and Validation  Workshop Report',
   'bbox': (56.692901611328125,
    651.9644775390625,
    395.2968444824219,
    750.0764770507812),
   'font_size': 28.0,
   'page_num': 0},
  {'text': 'May 2025',
   'bbox': (56.692901611328125,
    780.1727905273438,
    112.79410552978516,
    796.5167846679688),
   'font_size': 12.0,
   'page_num': 0},
  {'text': 'SAVING LIVES CHANGING LIVES',
   'bbox': (474.7085876464844,
    618.0900268554688,
    525.9020385742188,
    667.380615234375),
   'font_size': 10.750399589538574,
   'page_num': 0},
  {'text': 'WFP SOMALIA',
   'bbox': (70.86599731445312,
    556.7276000976562,
    193.24349975585938,
    580.5626220703125),
   'font_size': 17.5,
   'page_num': 0}],
 [{'text': 'Anticipatory Action Learning and Validation Workshop Report 2',
   'bbox': (56.692901611328125,
    787.1781616210938,
    538.5841674804688,
    797.826171875),
   'font_size': 8.0,
   'page_num': 1},
  {'text': '1.\t Introduction',
 